# Deploy MiniMax-H3 on Amazon SageMaker AI

[MiniMax-H3](https://huggingface.co/MiniMaxAI/MiniMax-H3) is an omni-modal generative system: it takes text, images, video and audio as context and generates **video with natively synchronized stereo audio** — up to 15 seconds, 24 fps, 32 kHz stereo.

This is not a chat model. If you arrived here from the LLM examples in this repo (Kimi, Qwen, GLM), almost none of the deployment shape carries over. The closest relative here is [`01-models/Wan/Wan2.1-T2V-1.3B-Diffusers`](../../Wan/Wan2.1-T2V-1.3B-Diffusers/), and even that differs in three material ways covered below.

| Property | Value |
| :--- | :--- |
| Model | [`MiniMaxAI/MiniMax-H3`](https://huggingface.co/MiniMaxAI/MiniMax-H3) |
| H3-Omni-Transformer | 33B dense, single-stream; ~13B of that in AdaLN branches |
| Text encoder | Qwen3-VL-32B (hidden states from layer 50) |
| Visual VAE | f16t4d24, temporally causal; patchify 1x2x2 |
| Audio VAE | 32 kHz in/out, 40 Hz latent, stereo |
| Precision | BF16 / FP32 mixed (FP32 preserved on patch, time and output projections) |
| Output | H.264 video @ 24 fps + AAC stereo @ 32 kHz, in one MP4 |
| Duration | 4-15 seconds inclusive |
| Resolution | 768-pixel short edge (2K only via the hosted Regenerate API) |
| License | **MiniMax H3 Community License** — not Apache 2.0 |

Architecture figures and the serving recipes below are quoted from the model card and the [SGLang MiniMax-H3 cookbook](https://docs.sglang.io/cookbook/diffusion/MiniMax/MiniMax-H3).


## Read this before you deploy

### 1. Only part of the system is open

The full H3 system is three modules. **One** of them is in this release.

| Module | Role | Open? |
| :--- | :--- | :--- |
| H3-Context-IR | Turns free-form multimodal input into the structured representation H3-Base consumes | **No** — hosted API |
| H3-Base | Generates 768p video + audio | **Yes** — this notebook |
| H3-Regenerate-2K | In-context regeneration to 2K | **No** — hosted API |

MiniMax describe H3-Context-IR as critical to output quality and strongly recommend either calling their API for it or building an equivalent from their prompting guide. A locally deployed H3-Base fed raw user prompts will underperform the hosted product, and that gap is a prompt-engineering gap, not a serving bug. Budget for it.

### 2. Two checkpoint partitions, two endpoints

`--model-variant` is fixed at server launch and selects which weights load:

| Partition | Serves tasks | Conditioning |
| :--- | :--- | :--- |
| `fl2va` | `t2va`, `fl2va` | none, or first/last keyframes |
| `ref2va` | `ref2va` (includes video-to-video) | images, videos, audio as references |

One endpoint cannot serve both. Covering all task modes means **two endpoints**. The serving shim in `container/` rejects a mismatched task with a clear error rather than silently producing wrong output.

### 3. Asynchronous inference, not real-time

A single 5-second 768p request takes tens of seconds of GPU time even on B300. Model load alone is roughly two minutes. This is the same reason the Wan example in this repo uses an async endpoint, and it applies more strongly here.

Async also happens to be the right *shape*: reference media arrives as S3 objects, output is a multi-megabyte MP4 that belongs in S3, and requests queue naturally.


## Instance selection

This is where H3 diverges most sharply from the Wan example.

SGLang published measured peak memory per GPU for the lossless BF16/FP32 path at 1344x768, 124 frames, 50 steps:

| Topology | Pipeline latency | Peak / GPU |
| :--- | ---: | ---: |
| 4x H100 — TP2 + Ulysses2 | 13.25 s | 66.04 GB |
| 4x H100 — FSDP + Ulysses4 | 13.36 s | 57.01 GB |
| 4x H100 — TP4 + Ulysses1 | 13.86 s | **49.80 GB** |

49.80 GB is the most memory-frugal measured Hopper topology. Against SageMaker instances:

| Instance | GPUs | HBM / GPU | Verdict |
| :--- | :--- | ---: | :--- |
| `ml.g6e.12xlarge` | 4x L40S | 48 GB | **Does not fit** — short by ~1.8 GB, and no NVLink |
| `ml.p5.48xlarge` | 8x H100 | 80 GB | Fits — matches the verified 4x H100 recipes |
| `ml.p5e.48xlarge` | 8x H200 | 141 GB | Fits — matches verified 4x H200 Ulysses4 |
| `ml.p6-b200.48xlarge` | 8x B200 | 180 GB | Fits — matches verified 8x B200 Ulysses8, and FP8 is only verified here |

**`ml.g6e.12xlarge` is the instance the Wan example uses and the natural first guess. It does not work for H3.** It misses on the single most favourable measured configuration, and L40S has no NVLink — Ulysses sequence parallelism is all-to-all heavy, so PCIe would hurt badly even if the memory fit. Do not start there.

Two further notes on topology:

- **Ulysses, not Ring.** Ring attention is incompatible with H3's packed multi-segment attention.
- **CFG parallelism is rejected outright.** The released checkpoints are CFG-distilled and run a single denoising branch; `--enable-cfg-parallel true` errors rather than silently duplicating work.


In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3 sagemaker huggingface_hub

In [ ]:
import json
import os
import time

import boto3
import sagemaker
from IPython.display import Video, display

sess = sagemaker.session.Session()
role = sagemaker.get_execution_role()
bucket = sess.default_bucket()
region = sess._region_name
account_id = sess.account_id()

sm = boto3.client("sagemaker")
smr = boto3.client("sagemaker-runtime")
s3 = boto3.client("s3")

print(f"role:   {role}")
print(f"bucket: {bucket}")
print(f"region: {region}")

In [ ]:
model_id = "MiniMaxAI/MiniMax-H3"

# fl2va serves t2va + fl2va; ref2va serves ref2va (incl. video-to-video).
MODEL_VARIANT = "fl2va"

instance = {"type": "ml.p5e.48xlarge", "num_gpu": 8}

# Topology per instance family, from the SGLang verified matrix.
# Only the flags below have completed a real request on that exact GPU model.
TOPOLOGY = {
    "ml.p5e.48xlarge":      "--num-gpus 4 --ulysses-degree 4",              # 4x H200 resident
    "ml.p5.48xlarge":       "--num-gpus 4 --tp-size 2 --ulysses-degree 2",  # 4x H100 fastest
    "ml.p6-b200.48xlarge":  "--num-gpus 8 --ulysses-degree 8",              # 8x B200 resident
}

s3_model_prefix = f"model/{model_id}"
model_name = f"minimax-h3-{MODEL_VARIANT}-{time.strftime('%y%m%d-%H%M%S')}"
endpoint_name = model_name
endpoint_config_name = model_name
variant_name = "main"

# Load is ~112-124s once weights are local, plus 26-39s warmup, plus the S3
# download of a very large checkpoint. Be generous; a tight value here is the
# most common cause of a failed deployment for this model.
startup_timeout = 3600

print(f"variant:  {MODEL_VARIANT}")
print(f"instance: {instance['type']}")
print(f"topology: {TOPOLOGY[instance['type']]}")

## 1. Stage the weights in S3

The repository hosts the original checkpoints (`FL2VA/`, `Ref2VA/`) **and** a diffusers-format copy side by side. Downloading the whole repo pulls far more than any one framework needs, so scope it — the model card is explicit about this.

Each partition carries its own transformer, the Qwen3-VL-32B text encoder, and both VAEs, so a single partition is still large. Staging in S3 once and pointing the container at it is much faster than pulling from the Hub on every endpoint creation.


In [ ]:
from huggingface_hub import snapshot_download

local_dir = f"./data/{MODEL_VARIANT}"

# Scope the download to one partition. Add "Ref2VA/*" when you deploy the
# second endpoint. Omitting allow_patterns pulls the diffusers copy too.
snapshot_download(
    repo_id=model_id,
    allow_patterns=[f"{MODEL_VARIANT.upper().replace('FL2VA', 'FL2VA').replace('REF2VA', 'Ref2VA')}/*"],
    local_dir=local_dir,
)
print("downloaded to", local_dir)

In [ ]:
# Upload with the CLI — far faster than per-file boto3 calls for a checkpoint
# this size, and it parallelises multipart uploads.
!aws s3 sync {local_dir} s3://{bucket}/{s3_model_prefix}/ --only-show-errors
print(f"staged at s3://{bucket}/{s3_model_prefix}/")

## 2. Build the serving container

There is no AWS Deep Learning Container carrying SGLang's diffusion extras today, so this example builds one. The image is the upstream SGLang dev image plus:

1. the platform-specific diffusion extra (absent from the base image — SGLang's own Docker recipe installs it at launch for the same reason),
2. `ffmpeg`, for muxing the H.264 + AAC output,
3. a Flask shim providing SageMaker's `/ping` and `/invocations` contract on port 8080.

The shim also does two things worth calling out, because both are easy to get wrong:

- **It rewrites `s3://` reference URIs to server-local `file://` paths.** SGLang resolves `conditions[].uri` inside its own filesystem, so an S3 URI passed straight through will fail.
- **It rejects a task that doesn't match the loaded partition**, instead of letting the request produce nonsense.

Files are in `container/`. Pin the base image to a digest before production use — `:dev` moves.


In [ ]:
ecr_repo = "minimax-h3-sglang"
image_tag = "0.1.0"
image_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{ecr_repo}:{image_tag}"

!aws ecr describe-repositories --repository-names {ecr_repo} >/dev/null 2>&1 || aws ecr create-repository --repository-name {ecr_repo} >/dev/null

!aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account_id}.dkr.ecr.{region}.amazonaws.com

!docker build --platform linux/amd64 -t {ecr_repo}:{image_tag} ./container
!docker tag {ecr_repo}:{image_tag} {image_uri}
!docker push {image_uri}

print(image_uri)

## 3. Create the model

`SGLANG_ARGS` carries the topology, so the same image serves every instance family and either partition without a rebuild.


In [ ]:
sglang_args = (
    f"--model-path /opt/ml/model "
    f"--model-variant {MODEL_VARIANT} "
    f"{TOPOLOGY[instance['type']]} "
    f"--performance-mode speed"
)

container_env = {
    "SGLANG_ARGS": sglang_args,
    "MODEL_VARIANT": MODEL_VARIANT,
    "OUTPUT_BUCKET": bucket,
    "OUTPUT_PREFIX": "minimax-h3/out",
    "JOB_TIMEOUT_S": "3000",
}

sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": image_uri,
        "ModelDataSource": {
            "S3DataSource": {
                "S3Uri": f"s3://{bucket}/{s3_model_prefix}/",
                "S3DataType": "S3Prefix",
                "CompressionType": "None",
            }
        },
        "Environment": container_env,
    },
)
print("created model:", model_name)
print("sglang args: ", sglang_args)

## 4. Create the async endpoint

`MaxConcurrentInvocationsPerInstance` is deliberately low. H3 at `--performance-mode speed` keeps every component resident and a single request already saturates the GPUs; queuing more in parallel raises peak memory without improving throughput. Raise it only after measuring.


In [ ]:
# Optional. Delete the NotificationConfig block if you don't want SNS.
sns_topic = "<YOUR_SNS_TOPIC>"

async_config = {
    "ClientConfig": {"MaxConcurrentInvocationsPerInstance": 1},
    "OutputConfig": {
        "S3OutputPath": f"s3://{bucket}/minimax-h3/async/out",
        "S3FailurePath": f"s3://{bucket}/minimax-h3/async/err",
        # "NotificationConfig": {
        #     "SuccessTopic": sns_topic,
        #     "ErrorTopic": sns_topic,
        #     "IncludeInferenceResponseIn": ["SUCCESS_NOTIFICATION_TOPIC"],
        # },
    },
}

sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": variant_name,
            "ModelName": model_name,
            "InstanceType": instance["type"],
            "InitialInstanceCount": 1,
            "ContainerStartupHealthCheckTimeoutInSeconds": startup_timeout,
            "ModelDataDownloadTimeoutInSeconds": startup_timeout,
        },
    ],
    AsyncInferenceConfig=async_config,
)

sm.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=endpoint_config_name)
sess.wait_for_endpoint(endpoint_name)

## 5. Inference

Async invocation takes its payload from S3, so each request is written to a key first. Two small helpers cover submit and collect.


In [ ]:
def submit(payload, name=None):
    """Write a request to S3 and invoke the async endpoint."""
    name = name or f"req-{time.strftime('%H%M%S')}-{payload.get('task', 't2va')}"
    key = f"minimax-h3/async/in/{name}.json"
    s3.put_object(Bucket=bucket, Key=key, Body=json.dumps(payload).encode())

    res = smr.invoke_endpoint_async(
        EndpointName=endpoint_name,
        ContentType="application/json",
        InputLocation=f"s3://{bucket}/{key}",
        InvocationTimeoutSeconds=3600,
    )
    print(f"submitted {name} -> {res['OutputLocation']}")
    return res["OutputLocation"]


def collect(output_location, poll_s=15, timeout_s=3600):
    """Poll the async OutputLocation until the response object appears."""
    path = output_location.replace("s3://", "")
    out_bucket, out_key = path.split("/", 1)
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            body = s3.get_object(Bucket=out_bucket, Key=out_key)["Body"].read()
            return json.loads(body)
        except s3.exceptions.NoSuchKey:
            print(".", end="", flush=True)
            time.sleep(poll_s)
    raise TimeoutError("no async response within timeout")


def fetch_local(s3_uri, dest=None):
    """Download a generated MP4 so it can be played inline."""
    path = s3_uri.replace("s3://", "")
    b, k = path.split("/", 1)
    dest = dest or os.path.basename(k)
    s3.download_file(b, k, dest)
    return dest

### Text to video and audio (`t2va`)

`flow_shift` governs the video diffusion trajectory and `audio_flow_shift` the audio one — they are separate knobs. The values below are the verified 5-second profile from the SGLang cookbook.


In [ ]:
t2va = {
    "bucket": bucket,
    "file_name": "h3-t2va-demo",
    "model": model_id,
    "task": "t2va",
    "prompt": (
        "A rain-slicked Tokyo alley at night. Neon signage reflects in standing "
        "water as a lone cyclist rides slowly past, tyres hissing on wet asphalt."
    ),
    "seconds": 5,
    "conditions": [],
    "target": {"short_edge": 768, "aspect_ratio": "16:9", "duration_seconds": 5.0},
    "num_outputs_per_prompt": 1,
    "num_inference_steps": 50,
    "flow_shift": 12.0,
    "audio_flow_shift": 3.0,
    "seed": 1101,
}

loc = submit(t2va)
result = collect(loc)
print(json.dumps(result, indent=2))

In [ ]:
local = fetch_local(result["outputs"][0])
display(Video(local, embed=True, width=768))

### First/last frame to video and audio (`fl2va`)

Supply one or two image conditions with role `keyframe`; valid `frame_index` sets are `[0]`, `[-1]`, and `[0, -1]`.

Choose `fl2va` when the supplied image should literally *be* the first or last frame. Use image-based `ref2va` instead when the image should guide identity or style without being preserved as an endpoint — `ref2va` may recompose or crop it.

The shim downloads `s3://` conditions to the server's filesystem before handing them to SGLang.


In [ ]:
# Upload a keyframe for conditioning.
first_frame_key = "minimax-h3/media/first-frame.png"
# s3.upload_file("first-frame.png", bucket, first_frame_key)

fl2va = {
    "bucket": bucket,
    "file_name": "h3-fl2va-demo",
    "model": model_id,
    "task": "fl2va",
    "prompt": "The scene continues with calm, natural motion and synchronized ambient sound.",
    "seconds": 5,
    "conditions": [
        {
            "type": "image",
            "uri": f"s3://{bucket}/{first_frame_key}",
            "role": "keyframe",
            "frame_index": 0,
        }
    ],
    "target": {"short_edge": 768, "aspect_ratio": "auto", "duration_seconds": 5.0},
    "num_outputs_per_prompt": 1,
    "num_inference_steps": 50,
    "flow_shift": 12.0,
    "audio_flow_shift": 3.0,
    "seed": 2101,
}

loc = submit(fl2va)
result = collect(loc)
print(json.dumps(result, indent=2))

### Reference to video and audio (`ref2va`)

**Requires a second endpoint** launched with `--model-variant ref2va`. Re-run the cells above with `MODEL_VARIANT = "ref2va"` and stage `Ref2VA/*`.

Reference limits from the model card: at most 9 images, 3 video clips, and 3 audio clips, with no more than 12 files total. Video and audio clips must each be 2-15 seconds, total duration at most 15 seconds, and audio cannot be the sole input.

Condition order is semantic. `<Picture 1>`, `<Video 1>`, `<Audio 1>` in the prompt refer to the one-based order **within each modality**, so the tags must line up with the order in `conditions`.


In [ ]:
ref2va = {
    "bucket": bucket,
    "file_name": "h3-ref2va-demo",
    "model": model_id,
    "task": "ref2va",
    "prompt": "Use <Picture 1> as the visual subject and <Audio 1> as the sound reference, with coherent natural motion.",
    "seconds": 5,
    "conditions": [
        {"type": "image", "uri": f"s3://{bucket}/minimax-h3/media/reference.png", "role": "reference"},
        {"type": "audio", "uri": f"s3://{bucket}/minimax-h3/media/reference.mp3", "role": "reference"},
    ],
    "target": {"short_edge": 768, "aspect_ratio": "auto", "duration_seconds": 5.0},
    "num_outputs_per_prompt": 1,
    "num_inference_steps": 50,
    "flow_shift": 12.0,
    "audio_flow_shift": 3.0,
    "seed": 3101,
}

# Submit against the ref2va endpoint, not the fl2va one.
# loc = submit(ref2va)
# result = collect(loc)
print("Deploy a --model-variant ref2va endpoint before running this cell.")

### Video to video

V2V is a `ref2va` use case, not a fourth task value — provide a video reference and keep `task` as `ref2va`.

Use `type: "video"` when the source may be silent; if it has a soundtrack H3 also treats that as an audio reference. Use `type: "video_audio"` only when both streams are required — that form rejects an input without audio.

`start_time_seconds` selects a segment from a longer source; the visual and audio streams are always sought together.

One expectation to set with customers: **`ref2va` treats the input video as reference material, not a pixel-aligned edit source.** It can resynthesize or reorder motion and cuts, and exposes no denoising-strength control. It is not a video-to-video editor in the img2img sense.


In [ ]:
v2v = {
    "bucket": bucket,
    "file_name": "h3-v2v-demo",
    "model": model_id,
    "task": "ref2va",
    "prompt": "Follow the motion and appearance of <Video 1>, changing the setting to a moonlit bedroom while preserving coherent timing.",
    "seconds": 5,
    "conditions": [
        {
            "type": "video",
            "uri": f"s3://{bucket}/minimax-h3/media/input.mp4",
            "role": "reference",
            "start_time_seconds": 35.0,
        }
    ],
    "target": {"short_edge": 768, "aspect_ratio": "16:9", "duration_seconds": 5.0},
    "num_outputs_per_prompt": 1,
    "num_inference_steps": 50,
    "flow_shift": 12.0,
    "audio_flow_shift": 3.0,
    "seed": 4101,
}
print("Submit against the ref2va endpoint.")

## 6. Quality profiles

`quality` is a **request-scoped** sampling parameter, so one resident endpoint can switch per request. An approximate profile mounts its Cache-DiT policy at the batch boundary; a later `lossless` request removes the hooks.

Measured on 4x H200 at 1344x768, 124 frames, 50 steps, averaged over three prompt/seed pairs:

| `quality` | Mean latency | Speedup | SSIM vs lossless | PSNR vs lossless |
| :--- | ---: | ---: | ---: | ---: |
| `lossless` | 75.10 s | 1.00x | 1.000 | exact |
| `high` | 53.70 s | 1.40x | 0.931 | 28.16 dB |
| `medium` | 30.23 s | 2.48x | 0.818 | 20.40 dB |
| `low` | 25.81 s | 2.91x | 0.794 | 19.25 dB |

Two caveats that matter when quoting these to a customer:

- SSIM and PSNR measure **trajectory deviation from the same seed**, not absolute perceptual quality. An approximate profile can produce a different but equally plausible result.
- Both metrics cover **video only**, while the profiles also change the joint audio-video denoise trajectory. Listen to the audio before shipping an approximate profile.

The named profiles are fail-closed to the audited 4x H200 workload above. Other hardware, task modes, step counts or flow shifts are rejected before denoising rather than silently degrading.


In [ ]:
preview = dict(t2va, file_name="h3-t2va-preview", quality="low", seed=1102)
# loc = submit(preview); result = collect(loc)
print(json.dumps({k: preview[k] for k in ("task", "quality", "seed")}, indent=2))

## 7. Scale to zero

An idle `ml.p5e.48xlarge` is expensive. Async endpoints support scaling to zero instances, with requests queueing until capacity returns — the natural fit for bursty generation traffic.

Note the tradeoff explicitly: a cold start pays the full model download plus roughly two minutes of load and warmup. Set `ScaleInCooldown` long enough that a burst does not repeatedly pay it.


In [ ]:
aas = boto3.client("application-autoscaling")
resource_id = f"endpoint/{endpoint_name}/variant/{variant_name}"

aas.register_scalable_target(
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    MinCapacity=0,
    MaxCapacity=2,
)

aas.put_scaling_policy(
    PolicyName="h3-backlog-scaling",
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    PolicyType="TargetTrackingScaling",
    TargetTrackingScalingPolicyConfiguration={
        "TargetValue": 1.0,
        "CustomizedMetricSpecification": {
            "MetricName": "ApproximateBacklogSizePerInstance",
            "Namespace": "AWS/SageMaker",
            "Dimensions": [{"Name": "EndpointName", "Value": endpoint_name}],
            "Statistic": "Average",
        },
        "ScaleInCooldown": 900,
        "ScaleOutCooldown": 300,
    },
)
print("registered scale-to-zero for", endpoint_name)

## Cleanup

In [ ]:
sess.delete_endpoint(endpoint_name)
sess.delete_endpoint_config(endpoint_config_name)
sess.delete_model(model_name)
print("deleted:", endpoint_name)